In [1]:
import numpy as np
import geopandas as gpd
import pandas as pd
from scipy import stats

In [2]:
datadir = '/projects/standard/lenkne/oboiko/EJ/data/'
regions = [
    'Headwaters', 'Gorge', 'Driftless', 'Working River', 'Confluence',
    'Chickasaw', 'Delta', 'Lower Mississippi', 'Gulf South', 'Total'
]
periods = ['2008_2012', '2013_2017', '2018_2022']

# load and merge the data
gdf = gpd.read_file(datadir + 'aoi_huc12_boundaries.gpkg')
huc12_rsei = pd.read_csv(
    datadir + 'huc12_rsei_toxconc_weighted.csv', dtype={'huc12': str}, index_col=0)
huc12_demographics = pd.read_csv(
    datadir + 'huc12_demographics.csv', dtype={'huc12': str}, index_col=0)
# combine tabular and spatial data using huc12 ids
gdf = gdf.merge(huc12_rsei, on='huc12',  suffixes=('', '_DROP'))
gdf = gdf.merge(huc12_demographics, on='huc12', suffixes=('', '_DROP'))
gdf = gdf.filter(regex='^(?!.*_DROP)')

ERROR 1: PROJ: proj_create_from_database: Open of /users/2/oboiko/.conda/envs/geo/share/proj failed


### Footprint of exposure

In [3]:
results = []
for region in regions:
    if region == 'Total':
        subset = gdf.copy()
    else:
        subset = gdf[gdf['Region'] == region].copy()
    for p in periods:
        tox_col, pop_col = f'{p}_TOXCONC', f'{p}_total'
        subset_notnull = subset.dropna(subset=[tox_col])
        pop_total = subset[pop_col].sum()
        pop_exposed = subset_notnull[pop_col].sum()
        results.append({
            'Region': region,
            'Period': p,
            'n_total': len(subset),
            'n_exposed':len(subset_notnull),
            'pop_total': pop_total,
            'pop_exposed': pop_exposed
        })
df_stats = pd.DataFrame(results)
df_stats['Region'] = pd.Categorical(df_stats['Region'], categories=regions, ordered=True)
# Pivot to MultiIndex and organize columns
df_pivot = df_stats.pivot(index='Region', columns='Period')
df_pivot = df_pivot.swaplevel(0, 1, axis=1).sort_index(axis=1)

In [4]:
df_pivot

Period            2008_2012                                     2013_2017  \
                  n_exposed n_total   pop_exposed     pop_total n_exposed   
Region                                                                      
Headwaters               20     314  5.353755e+04  2.083553e+05        20   
Gorge                    53     240  1.533588e+06  3.359793e+06        54   
Driftless               112     430  4.114985e+05  7.087400e+05       117   
Working River           108     390  5.183283e+05  8.522516e+05       108   
Confluence              124     344  1.674937e+06  2.979302e+06       120   
Chickasaw                56     111  1.544730e+05  2.726939e+05        54   
Delta                   151     365  1.046688e+06  1.559113e+06       164   
Lower Mississippi        40     189  3.816238e+04  2.214515e+05        42   
Gulf South               81     193  6.829645e+05  1.794739e+06        77   
Total                   745    2576  6.114177e+06  1.195644e+07       756   

Period                                                2018_2022          \
                  n_total   pop_exposed     pop_total n_exposed n_total   
Region                                                                    
Headwaters            314  5.393438e+04  2.088459e+05        20     314   
Gorge                 240  1.782736e+06  3.523208e+06        50     240   
Driftless             430  4.091806e+05  7.079433e+05       116     430   
Working River         390  5.139523e+05  8.513441e+05       102     390   
Confluence            344  1.683285e+06  2.995549e+06       106     344   
Chickasaw             111  1.589690e+05  2.629144e+05        51     111   
Delta                 365  1.154768e+06  1.564966e+06       168     365   
Lower Mississippi     189  5.884876e+04  2.157320e+05        32     189   
Gulf South            193  5.698186e+05  1.876681e+06        77     193   
Total                2576  6.385493e+06  1.220718e+07       722    2576   

Period                                         
                    pop_exposed     pop_total  
Region                                         
Headwaters         5.698862e+04  2.136152e+05  
Gorge              1.860536e+06  3.681148e+06  
Driftless          4.116651e+05  7.107090e+05  
Working River      5.114122e+05  8.448692e+05  
Confluence         1.560298e+06  2.986926e+06  
Chickasaw          1.445194e+05  2.513851e+05  
Delta              1.232690e+06  1.536235e+06  
Lower Mississippi  5.017896e+04  2.096893e+05  
Gulf South         5.928496e+05  1.871014e+06  
Total              6.421138e+06  1.230559e+07

In [6]:
#df_pivot.to_csv(datadir + 'huc12_toxconc_descriptive_stats.csv')

### T-Test on impacted vs non-impacted watersheds

In [23]:
# Keep base variable names clean (no string formatting syntax inside the list)
base_variables = [
    'share_nonhsp_white', 'share_black', 'share_native',
    'share_asian', 'share_hispanic', 'share_below_poverty', 'share_2_above_poverty'
]

results = {}

for period in periods:
    print(f"Processing: {period}")
    
    # Map base variables to their period-specific column names
    var_cols = [f"{period}_{v}" for v in base_variables]
    tox_col = f"{period}_TOXCONC"
    
    # Split the DataFrame ONCE per period
    is_impacted = gdf[tox_col].notnull()
    df_imp = gdf.loc[is_impacted, var_cols]
    df_nonimp = gdf.loc[~is_impacted, var_cols] # ~ means "not", which is safer and cleaner
    
    # Calculate all means at once (vectorized!)
    avg_imp = df_imp.mean() * 100
    avg_nonimp = df_nonimp.mean() * 100
    
    # Run t-tests across columns simultaneously
    t_stats, p_vals = stats.ttest_ind(df_imp, df_nonimp, equal_var=False, nan_policy='omit')
    
    # Store results using the actual column names as keys
    for i, col in enumerate(var_cols):
        results[col] = {
            'avg_impacted, %': avg_imp[col],
            'avg_nonimpacted, %': avg_nonimp[col],
            't_stat': t_stats[i],
            'p_val': p_vals[i]
        }

# Convert the structured dictionary directly into your final DataFrame
ttest_summary = pd.DataFrame.from_dict(results, orient='index')
ttest_summary

Processing: 2008_2012
Processing: 2013_2017
Processing: 2018_2022


,"avg_impacted, %","avg_nonimpacted, %",t_stat,p_val
2008_2012_share_nonhsp_white,80.767998,84.726037,-4.093374,4.507171e-05
2008_2012_share_black,14.926342,10.853484,4.265862,2.133921e-05
2008_2012_share_native,0.339902,1.202041,-6.595322,5.335455e-11
2008_2012_share_asian,0.598100,0.475771,2.316124,2.071557e-02
2008_2012_share_hispanic,2.350984,1.759461,4.546101,6.027274e-06
2008_2012_share_below_poverty,14.344568,13.710523,1.575300,1.154259e-01
2008_2012_share_2_above_poverty,65.027352,65.722860,-1.108510,2.678490e-01
2013_2017_share_nonhsp_white,79.471290,84.303404,-4.950358,8.346664e-07
2013_2017_share_black,15.721733,10.864699,5.000817,6.474064e-07
2013_2017_share_native,0.285293,1.146148,-7.180140,9.894073e-13


In [24]:
#ttest_summary.to_csv(datadir + 'huc12_toxconc_ttest_stats.csv')

### Spatial agreement between impacted watersheds across time

In [10]:
# Define your columns
cols = ['2008_2012_TOXCONC', '2013_2017_TOXCONC', '2018_2022_TOXCONC']
# Create a boolean mask: True if impacted (not Null), False if Null
impact_mask = gdf[cols].notnull()
# Calculate Intersection (rows where ALL columns are True)
intersection = impact_mask.all(axis=1).sum()
# Calculate Union (rows where AT LEAST ONE column is True)
union = impact_mask.any(axis=1).sum()
# Compute Jaccard Index
jaccard_score = intersection / union if union > 0 else 0
print(f"Global Jaccard Score: {jaccard_score:.4f}")
print(f"Agreement Area: {intersection} HUC-12s")
print(f"Total Impacted Footprint: {union} HUC-12s")

Global Jaccard Score: 0.8185
Agreement Area: 663 HUC-12s
Total Impacted Footprint: 810 HUC-12s
